In [1]:
import pandas as pd
import numpy as np
import os

In [4]:
data_path = "../../data/Georgios/"

frequency_path = os.path.join(data_path, "user_product_frequency.csv") # Bui/bu
recency_path = os.path.join(data_path, "user_product_recency_normalized.csv") # e^-λ
tfidf_path = os.path.join(data_path, "user_product_tfidf_normalized.csv")

frequency = pd.read_csv(frequency_path)
recency = pd.read_csv(recency_path)
tfidf = pd.read_csv(tfidf_path)

In [13]:
merged = (
    frequency.merge(recency, on=['user_id', 'product_id'], how='inner')
                .merge(tfidf, on=['user_id', 'product_id'], how='inner')
)

merged = merged.rename(columns={
    'score':'recency_score', # e^-λ
    'normalized_score' : 'normalized_recency',
    'tfidf_score' : 'normalized_tfidf'
    })

print(f"Merged shape: {merged.shape}")
merged.head()

Merged shape: (13307953, 8)


,user_id,product_id,Bui,Bu,freq_ui,recency_score,normalized_recency,normalized_tfidf
0,1,196,10,10,1.0,4.192653,0.050238,0.052693
1,1,10258,9,10,0.9,4.304592,0.051580,0.086337
2,1,10326,1,10,0.1,0.398121,0.004770,0.007548
3,1,12427,10,10,1.0,4.570855,0.054770,0.078029
4,1,13032,3,10,0.3,1.765810,0.021159,0.024680


In [ ]:
w_freq, w_rec, w_tfidf = (1/3), (1/3), (1/3)
merged['ranke_ui'] = (
    w_freq * merged['freq_ui'] +
    w_rec * merged['normalized_recency'] +
    w_tfidf * merged['normalized_tfidf']
)

final_ratings = merged[['user_id', 'product_id', 'ranke_ui']]

filename_csv = f"ratings_{w_freq:.2f}_{w_rec:.2f}_{w_tfidf:.2f}.csv"
filename_parquet = f"ratings_{w_freq:.2f}_{w_rec:.2f}_{w_tfidf:.2f}.parquet"

new_data_path = os.path.join(data_path, "RankedTableResults")
os.makedirs(new_data_path, exist_ok=True)

ratings_csv = os.path.join(new_data_path, filename_csv)
ratings_parquet = os.path.join(new_data_path, filename_parquet)

final_ratings.to_csv(ratings_csv, index=False)
final_ratings.to_parquet(ratings_parquet, index=False)
print(f"Saved ratings to:\n - {ratings_csv}\n - {ratings_parquet}")